In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/DIANNE')
import viewer
import dianne
import pandas as pd
import matplotlib.colors as mcolors
dianne.setNotebookWidth(100)

In [ ]:
dataPath = '/projects/activities/kappsen-tmc/USERS/domans/wsi-viewer/data/'

# Load the data and prepare the patches for annotation and training the classifier
samples = ['JDC-WP-012-w-STQ', 'JDC-WP-012-ae-STQ']
classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)
fname = 'img.data.ctranspath-1.h5ad'

# Load the demo Xenium data and cell annotations
all_annotations = {sample: None for sample in samples}
xenium_bundle_paths = {'JDC-WP-012-w-STQ':'/projects/activities/kappsen-tmc/USERS/domans/xe/spaceranger-custom-seg-outs/JDC-WP-012-w/outs', 'JDC-WP-012-ae-STQ': None}
xenium_to_he_matrices = {sample: f'{dataPath}/{sample}/matrix.csv' for sample in samples}
dataSubPathsXeniumSlim = ['xenium-slim-JDC-WP-012-w']
for xpath in dataSubPathsXeniumSlim:
    dianne.downloadFromZenodo(dataPath + xpath, 'https://zenodo.org/api/records/19240145/files-archive')
annotations = {sample: pd.read_csv(xenium_bundle_paths[sample] + '/annotated_fine.csv', index_col=0)['group'] for sample in ['JDC-WP-012-w-STQ']}
uannotations = sorted(set(a for sample in ['JDC-WP-012-w-STQ'] for a in annotations[sample].unique()))
all_annotations['JDC-WP-012-w-STQ'] = annotations['JDC-WP-012-w-STQ'].reset_index(drop=True).to_dict()
annotationsPalette = {a: mcolors.to_hex(dianne.Set123(i)) for i, a in enumerate(uannotations)}

ts, mpp = 56., 0.25 # center-to-center tile distance in um and pixel size
patch_size = 8 # number of tiles, in each dimension, to include in a patch (e.g. 8 means 8x8=64 tiles per patch)
ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne.loadDataAndPreparePatches(samples, dataPath, fname, L=None, ts=ts, mpp=mpp, N=patch_size)

In [ ]:
drawings = viewer.create_viewer(samples, imgs, height="900px", xenium_mpp=0.2125, max_cells=1000,
                                            matrices=xenium_to_he_matrices, xenium_bundle_paths=xenium_bundle_paths,
                                            annotations=all_annotations, category_colors=annotationsPalette)[1]

In [ ]:
if False:
    clf = dianne.loadGUIClassifier(classifierPaths, 'sma-gui-pancreas')
else:
    # Train the classifier on the patches and the corresponding features
    body_overlap = 0.25 # Fraction overlap between the drawn annotations and the tiles
    tile_size = 224 # pixels
    clf, patchesCDFsMod, annotationsMod = dianne.getClassifierForFromStrokes(drawings, patchCoordinates, tile_size, body_overlap, patch_size,
                                                                            ads, samples, qs, augFunc=dianne.PCMA, alpha=0.8, seed=0, showPatches=False)
# Run inference and visualize the results
if not clf is None:
    infSample = samples[1]
    multiplier = 2 # Interpolation parameter for the probability heatmap
    x, y, p = dianne.inferProb(ads[infSample], clf, qs, tsize=ts/mpp, R=2, erode=False)
    xi, yi, pi = dianne.interpolatePoints(x, y, p, multiplier=multiplier)
    # dianne.showProbImg(xi, yi, pi, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=samples[0], invert=False)
    viewer.set_overlay_points(xi, yi, pi, infSample, delta=224 / multiplier, alpha=0.5, color_low='#FFA500', color_high='#0000FF')

In [ ]:
dianne.saveGUIClassifier(clf, classifierPaths, 'sma-gui-pancreas', samples, ts, mpp, patch_size, tile_size, body_overlap, qs, patchesCDFsMod, annotationsMod, drawings)

In [ ]:
# Create probability masks, extract contours and visualize them on the H&E image
downsampled_map, fshape = dianne.makeProbMask(ads[infSample], imgs[infSample], x, y, p, ts=ts, mpp=mpp, downfactor=16, verbose=True)
geojson = dianne.extractContoursForQuPath(downsampled_map, fshape, cutoff=0.8, min_area=10**6, downfactor=16, sigma=100)
dianne.viewContoursOnImage(imgs[infSample], geojson, fshape, level=2, figsize=(12, 12), linewidth=1)